In [6]:
import numpy as np
import pandas as pd
import pywt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

# ---------- 0. Settings ----------
raw_interval_min = 1
bin_min = 5
dt = bin_min / 60                          # hours per row after binning
per_day = 24 * 60 // bin_min               # 288
wavelet = "cmor1.5-1.0"
periods = np.geomspace(1, 39, 100)
scales = periods / (pywt.central_frequency(wavelet) * dt)
cr_rows = (periods >= 23) & (periods <= 25)
ur_rows = (periods >= 1) & (periods <= 3)

# ---------- 1. Load ----------
female_temp = pd.read_csv('../data/inputs/fem_temp.csv')
female_act  = pd.read_csv('../data/inputs/fem_act.csv')

# ---------- 2. Clean and bin ----------
def prep_temp(df):
    df = df.select_dtypes("number").reset_index(drop=True)
    df = df.interpolate(limit=10).bfill().ffill()
    df = df.clip(lower=35)
    mu, sd = df.mean(), df.std()
    df = df.clip(lower=mu - 3*sd, upper=mu + 3*sd, axis=1)
    return df.groupby(df.index // (bin_min // raw_interval_min)).mean()

def bin_act(df):
    df = df.select_dtypes("number").reset_index(drop=True).fillna(0)
    return df.groupby(df.index // (bin_min // raw_interval_min)).mean()

f = prep_temp(female_temp)
act5 = bin_act(female_act)
n = min(len(f), len(act5))
f, act5 = f.iloc[:n], act5.iloc[:n]
assert f.shape[1] == act5.shape[1], "temp and activity sheets have different numbers of mice"
mice = f.shape[1]
print("rows:", n, "| mice:", mice, "| days:", round(n / per_day, 1))

# ---------- 3. Activity-corrected CBT ----------
resid = {}
for col in f.columns:
    a = np.log1p(act5[col].values)
    X = np.column_stack([np.roll(a, k) for k in (0, 1, 2, 3)])
    y_t = f[col].values
    resid[col] = y_t - LinearRegression().fit(X, y_t).predict(X) + y_t.mean()
resid = pd.DataFrame(resid, index=f.index)

# ---------- 4. Wavelet band amplitude (one transform per mouse) ----------
def band_series(df):
    """Returns (circadian, ultradian), each (mice, 12*per_day): transform on all
    14 days, first and last day cut, max amplitude in the band at each time point."""
    circ, ultr = [], []
    for col in df.columns:
        x = df[col].values - df[col].values.mean()
        coefs, _ = pywt.cwt(x, scales, wavelet, sampling_period=dt)
        amp = np.abs(coefs)[:, per_day : 13 * per_day]
        circ.append(amp[cr_rows].max(axis=0))
        ultr.append(amp[ur_rows].max(axis=0))
    return np.array(circ), np.array(ultr)

circ_raw, ultr_raw = band_series(f)
circ_adj, ultr_adj = band_series(resid)

# ---------- 5. Day-level features: one row per mouse per day ----------
def days(x):                                   # (mice, 12*per_day) -> (mice, 12, per_day)
    return x.reshape(mice, 12, per_day)

raw12 = days(f.values[per_day:13*per_day].T)
adj12 = days(resid.values[per_day:13*per_day].T)
act12 = days(np.log1p(act5.values)[per_day:13*per_day].T)

feats = {
    "ultr_raw":    days(ultr_raw).mean(2),
    "circ_raw":    days(circ_raw).mean(2),
    "ultr_adj":    days(ultr_adj).mean(2),
    "circ_adj":    days(circ_adj).mean(2),
    "range_raw":   raw12.max(2) - raw12.min(2),
    "plateau_raw": (raw12 > np.median(raw12, axis=(1, 2), keepdims=True)).sum(2) * dt,
    "range_adj":   adj12.max(2) - adj12.min(2),
    "act_total":   act12.mean(2),
    "act_dark":    act12[:, :, :per_day // 2].mean(2),   # assumes lights-off start (unverified)
}

CENTER_PER_MOUSE = True     # subtract each mouse's own 12-day mean: focuses on day-to-day change
if CENTER_PER_MOUSE:
    feats = {k: v - v.mean(axis=1, keepdims=True) for k, v in feats.items()}

# ---------- 6. Labels: days 1, 5, 9 = estrus (assumed from the paper) ----------
y = np.tile((np.arange(12) % 4 == 0).astype(int), mice)    # 39 positives, 117 negatives
groups = np.repeat(np.arange(mice), 12)                    # which mouse each row belongs to

def table(names):
    return np.column_stack([feats[k].reshape(-1) for k in names])

sets = {
    "plateau (circular)": ["plateau_raw", "range_raw"],
    "raw wavelet":        ["ultr_raw", "circ_raw"],
    "residual wavelet":   ["ultr_adj", "circ_adj", "range_adj"],
    "activity only":      ["act_total", "act_dark"],
    "everything":         list(feats),
}

# ---------- 7. Leave-one-mouse-out classification ----------
model = make_pipeline(StandardScaler(),
                      LogisticRegression(class_weight="balanced", max_iter=1000))
logo = LeaveOneGroupOut()

def evaluate(Xm, labels):
    p = cross_val_predict(model, Xm, labels, groups=groups, cv=logo,
                          method="predict_proba")[:, 1]
    return roc_auc_score(labels, p), balanced_accuracy_score(labels, (p > 0.5).astype(int))

# chance baseline: shuffle the estrus label across days within each mouse
rng = np.random.default_rng(0)
def shuffled_labels():
    return np.concatenate([rng.permutation(y[groups == g]) for g in range(mice)])

results = []
for name, cols in sets.items():
    Xm = table(cols)
    auc, bacc = evaluate(Xm, y)
    null = [evaluate(Xm, shuffled_labels())[0] for _ in range(20)]
    results.append((name, auc, bacc, np.mean(null), np.max(null)))

res = pd.DataFrame(results, columns=["feature set", "AUC", "balanced acc",
                                     "AUC shuffled (mean)", "AUC shuffled (max)"])
print(res.round(2).to_string(index=False))
print("\nchance AUC = 0.50; always guessing 'not estrus' = %.0f%% accuracy" % (100 * (1 - y.mean())))

# ---------- 8. Which features matter (full model, standardized coefficients) ----------
all_cols = list(feats)
full = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
full.fit(table(all_cols), y)
coef = pd.Series(full[-1].coef_[0], index=all_cols).sort_values(key=np.abs, ascending=False)
print("\nStandardized coefficients (positive = higher on estrus days):")
print(coef.round(2).to_string())

rows: 4032 | mice: 13 | days: 14.0
       feature set  AUC  balanced acc  AUC shuffled (mean)  AUC shuffled (max)
plateau (circular) 0.95          0.86                 0.46                0.57
       raw wavelet 0.93          0.85                 0.49                0.60
  residual wavelet 0.91          0.82                 0.48                0.64
     activity only 0.95          0.85                 0.52                0.66
        everything 0.99          0.94                 0.48                0.61

chance AUC = 0.50; always guessing 'not estrus' = 75% accuracy

Standardized coefficients (positive = higher on estrus days):
range_adj      1.68
ultr_raw      -1.57
act_dark       1.43
plateau_raw    1.23
ultr_adj      -0.64
circ_raw      -0.41
range_raw      0.35
act_total     -0.08
circ_adj      -0.04


In [8]:
def days(x):                                   # (mice, 12*per_day) -> (mice, 12, per_day)
    return x.reshape(mice, 12, per_day)

C_raw, U_raw = days(circ_raw), days(ultr_raw)
C_adj, U_adj = days(circ_adj), days(ultr_adj)
T_raw = days(f.values[per_day:13*per_day].T)
T_adj = days(resid.values[per_day:13*per_day].T)
A_log = days(np.log1p(act5.values)[per_day:13*per_day].T)

half = per_day // 2                            # 144 slots = 12 h
rows = []
for i, col in enumerate(f.columns):
    for d in range(12):
        rows.append({
            "mouse_id": col,
            "day_index": d + 2,                # 2..13, same numbering as the other file
            "vivian__cbt_circadian_amp_raw":  C_raw[i, d].mean(),
            "vivian__cbt_circadian_amp_adj":  C_adj[i, d].mean(),
            "vivian__cbt_ultradian_amp_raw":  U_raw[i, d].mean(),
            "vivian__cbt_ultradian_amp_adj":  U_adj[i, d].mean(),
            "vivian__circadian_retained":     C_adj[i, d].mean() / C_raw[i, d].mean(),
            "vivian__ultradian_retained":     U_adj[i, d].mean() / U_raw[i, d].mean(),
            "vivian__cbt_range_raw":          T_raw[i, d].max() - T_raw[i, d].min(),
            "vivian__cbt_range_adj":          T_adj[i, d].max() - T_adj[i, d].min(),
            "vivian__cbt_hours_above_own_median_raw": (T_raw[i, d] > np.median(T_raw[i])).sum() * dt,
            "vivian__ultradian_amp_raw_h0_12": U_raw[i, d, :half].mean(),
            "vivian__ultradian_amp_adj_h0_12": U_adj[i, d, :half].mean(),
            "vivian__la_log_mean":            A_log[i, d].mean(),
            "vivian__la_log_mean_h0_12":      A_log[i, d, :half].mean(),
        })

vivian = pd.DataFrame(rows)
vivian["estrus"] = ((vivian.day_index - 2) % 4 == 0).astype(int)   # days 2, 6, 10
vivian.to_csv("vivian_features.csv", index=False)
print(vivian.shape)          # expect (156, 16)
print(vivian.head(12).round(3))

(156, 16)
   mouse_id  day_index  vivian__cbt_circadian_amp_raw  \
0        f1          2                          8.213   
1        f1          3                          8.288   
2        f1          4                          8.093   
3        f1          5                          7.909   
4        f1          6                          8.089   
5        f1          7                          8.364   
6        f1          8                          8.427   
7        f1          9                          8.356   
8        f1         10                          8.282   
9        f1         11                          7.864   
10       f1         12                          7.293   
11       f1         13                          6.397   

    vivian__cbt_circadian_amp_adj  vivian__cbt_ultradian_amp_raw  \
0                           2.923                          0.561   
1                           2.908                          0.863   
2                           2.737           